In [20]:
#| default_exp transform.trim_data

In [21]:
#| export
from __future__ import annotations

import logging

import pandas as pd

logger = logging.getLogger("myproj.transform")

# Transform I: Spaltenauswahl

Aus den rohen Datensätzen werden nur die für die Analyse relevanten Spalten behalten. Alle anderen Spalten werden entfernt.

## Setup

In [22]:
from myproj.pipeline import run_import

## Daten laden

In [23]:
emdat_raw, sea_level_raw = run_import()

print(f"EMDAT: {emdat_raw.shape[0]:,} Zeilen, {emdat_raw.shape[1]} Spalten")
print(f"Sea Level: {sea_level_raw.shape[0]:,} Zeilen, {sea_level_raw.shape[1]} Spalten")

2026-05-12 23:36:04 | myproj.pipeline      | INFO     | run_import | start
2026-05-12 23:36:04 | myproj.io            | INFO     | Lade Rohdatei: public_emdat_1991_2024.xlsx
2026-05-12 23:36:08 | myproj.io            | INFO     | load_raw_data | file: public_emdat_1991_2024.xlsx | rows: 20657 | cols: 47
2026-05-12 23:36:08 | myproj.io            | INFO     | Lade Rohdatei: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc
2026-05-12 23:36:08 | myproj.io            | INFO     | load_raw_data | file: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc | variables: 2 | dimensions: {'time': 9405}
2026-05-12 23:36:08 | myproj.pipeline      | INFO     | run_import | done


EMDAT: 20,657 Zeilen, 47 Spalten
Sea Level: 9,405 Zeilen, 3 Spalten


## Spaltenanalyse
### EMDAT

In [24]:
print(f"Anzahl EMDAT-Spalten: {len(emdat_raw.columns)}")
for idx, col in enumerate(emdat_raw.columns, start=1):
    print(f"{idx:2d}. {col}")

Anzahl EMDAT-Spalten: 47
 1. DisNo.
 2. Historic
 3. Classification Key
 4. Disaster Group
 5. Disaster Subgroup
 6. Disaster Type
 7. Disaster Subtype
 8. External IDs
 9. Event Name
10. ISO
11. Country
12. Subregion
13. Region
14. Location
15. Origin
16. Associated Types
17. OFDA/BHA Response
18. Appeal
19. Declaration
20. AID Contribution ('000 US$)
21. Magnitude
22. Magnitude Scale
23. Latitude
24. Longitude
25. River Basin
26. Start Year
27. Start Month
28. Start Day
29. End Year
30. End Month
31. End Day
32. Total Deaths
33. No. Injured
34. No. Affected
35. No. Homeless
36. Total Affected
37. Reconstruction Costs ('000 US$)
38. Reconstruction Costs, Adjusted ('000 US$)
39. Insured Damage ('000 US$)
40. Insured Damage, Adjusted ('000 US$)
41. Total Damage ('000 US$)
42. Total Damage, Adjusted ('000 US$)
43. CPI
44. Admin Units
45. GADM Admin Units
46. Entry Date
47. Last Update


#### Keep oder Drop Entscheidungen

Die folgenden Ausgaben dokumentieren die Datenstruktur und inhaltlichen Eigenschaften der Spalten. Darauf basiert die Keep oder Drop Tabelle im Anschluss.

In [25]:
def _sample_values(series, n=3, max_len=90):
    vals = series.dropna().astype(str).unique()[:n]
    cleaned = [v.replace("\n", " ")[:max_len] for v in vals]
    return " | ".join(cleaned)

emdat_profile = pd.DataFrame(
    {
        "#": range(len(emdat_raw.columns)),
        "Spalte": emdat_raw.columns,
        "dtype": [str(emdat_raw[c].dtype) for c in emdat_raw.columns],
        "Nicht Null (%)": [round(emdat_raw[c].notna().mean() * 100, 1) for c in emdat_raw.columns],
        "Einzigartige Werte": [int(emdat_raw[c].nunique(dropna=True)) for c in emdat_raw.columns],
        "Beispielwerte": [_sample_values(emdat_raw[c]) for c in emdat_raw.columns],
    }
)

display(emdat_profile)

,#,Spalte,dtype,Nicht Null (%),Einzigartige Werte,Beispielwerte
0,0,DisNo.,str,100.0,20657,2018-0040-BRA | 2002-0351-USA | 1990-9604-BWA
1,1,Historic,str,100.0,2,No | Yes
2,2,Classification Key,str,100.0,65,nat-hyd-flo-flo | nat-cli-wil-for | nat-cli-dr...
3,3,Disaster Group,str,100.0,2,Natural | Technological
4,4,Disaster Subgroup,str,100.0,9,Hydrological | Climatological | Geophysical
5,5,Disaster Type,str,100.0,31,Flood | Wildfire | Drought
6,6,Disaster Subtype,str,100.0,65,Flood (General) | Forest fire | Drought
7,7,External IDs,str,21.2,3238,DFO:4576 | HANZE:21228 | HANZE:20549
8,8,Event Name,str,32.4,2822,Helicopter | Undine | Footbal stadium
9,9,ISO,str,100.0,226,BRA | USA | BWA


In [26]:
classification_cols = [
    "Historic",
    "Classification Key",
    "Disaster Group",
    "Disaster Subgroup",
    "Disaster Type",
    "Disaster Subtype",
    "Associated Types",
    "Origin",
]

for c in classification_cols:
    print(f"\n{c}: Top 12 Ausprägungen")
    vc = emdat_raw[c].value_counts(dropna=False).head(12)
    display(vc.to_frame(name="Anzahl"))

text_like_drop_cols = ["Event Name", "Location", "External IDs", "Admin Units", "GADM Admin Units"]
text_examples = []
for c in text_like_drop_cols:
    s = emdat_raw[c].dropna().astype(str)
    text_examples.append(
        {
            "Spalte": c,
            "Nicht Null (%)": round(emdat_raw[c].notna().mean() * 100, 1),
            "Mittlere Textlänge": round(s.str.len().mean(), 1) if len(s) else 0,
            "Beispiel": _sample_values(emdat_raw[c], n=2, max_len=120),
        }
    )

display(pd.DataFrame(text_examples))


Historic: Top 12 Ausprägungen


,Anzahl
Historic,
No,16153
Yes,4504



Classification Key: Top 12 Ausprägungen


,Anzahl
Classification Key,
tec-tra-roa-roa,2677
nat-hyd-flo-riv,2452
nat-met-sto-tro,1724
nat-hyd-flo-flo,1559
tec-tra-wat-wat,1424
nat-geo-ear-gro,854
nat-hyd-flo-fla,840
tec-tra-air-air,703
nat-bio-epi-bac,652



Disaster Group: Top 12 Ausprägungen


...


Associated Types: Top 12 Ausprägungen


,Anzahl
Associated Types,
NaN,16872
"Slide (land, mud, snow, rock)",1201
Flood,620
"Flood|Slide (land, mud, snow, rock)",322
Food shortage,130
Broken Dam/Burst bank,127
Rain,118
Hail,115
Flood|Hail,74



Origin: Top 12 Ausprägungen


,Anzahl
Origin,
NaN,16087
Heavy rains,1886
Heavy rain,661
Monsoonal rain,190
Torrential rains,181
Torrential rain,118
Brief torrential rain,110
Extreme rain,47
El Nino,44


,Spalte,Nicht Null (%),Mittlere Textlänge,Beispiel
0,Event Name,32.4,13.0,Helicopter | Undine
1,Location,94.1,60.2,Rio de Janeiro | Colorado province
2,External IDs,21.2,23.3,DFO:4576 | HANZE:21228
3,Admin Units,40.7,226.3,"[{""adm2_code"":9961,""adm2_name"":""Rio De Janeiro..."
4,GADM Admin Units,37.6,1563.7,"[{""gid_2"":""BRA.19.68_2"",""migration_date"":""2025..."


In [27]:
impact_cols = ["Total Deaths", "No. Injured", "No. Affected", "No. Homeless", "Total Affected"]
impact_num = emdat_raw[impact_cols].apply(pd.to_numeric, errors="coerce")

impact_quality = pd.DataFrame(
    {
        "Spalte": impact_cols,
        "Nicht Null (%)": [round(impact_num[c].notna().mean() * 100, 1) for c in impact_cols],
        "Median": [impact_num[c].median() for c in impact_cols],
        "95% Quantil": [impact_num[c].quantile(0.95) for c in impact_cols],
    }
)
display(impact_quality)

components = impact_num[["No. Injured", "No. Affected", "No. Homeless"]].fillna(0).sum(axis=1)
comparison = pd.DataFrame(
    {
        "Total Affected": impact_num["Total Affected"],
        "Summe Teilkomponenten": components,
    }
).dropna(subset=["Total Affected"])
comparison["Differenz"] = comparison["Total Affected"] - comparison["Summe Teilkomponenten"]

print(f"Zeilen mit vorhandenem Total Affected: {len(comparison):,}")
print(f"Anteil Total Affected >= Summe Teilkomponenten: {(comparison['Differenz'] >= 0).mean() * 100:.1f}%")
print(f"Anteil Total Affected == Summe Teilkomponenten: {(comparison['Differenz'] == 0).mean() * 100:.1f}%")
display(comparison.describe().T)

,Spalte,Nicht Null (%),Median,95% Quantil
0,Total Deaths,80.5,18.0,200.0
1,No. Injured,36.3,28.0,976.4
2,No. Affected,44.3,6631.0,1800000.0
3,No. Homeless,9.8,2100.0,152500.0
4,Total Affected,72.7,1000.0,806776.6


Zeilen mit vorhandenem Total Affected: 15,009
Anteil Total Affected >= Summe Teilkomponenten: 100.0%
Anteil Total Affected == Summe Teilkomponenten: 100.0%


,count,mean,std,min,25%,50%,75%,max
Total Affected,15009.0,446402.122793,5.884488e+06,1.0,50.0,1000.0,18016.0,330000000.0
Summe Teilkomponenten,15009.0,446402.122793,5.884488e+06,1.0,50.0,1000.0,18016.0,330000000.0
Differenz,15009.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0


In [28]:
geo_time_cols = [
    "ISO", "Country", "Region", "Subregion", "Latitude", "Longitude", "River Basin",
    "Start Year", "Start Month", "Start Day", "End Year", "End Month", "End Day",
    "Entry Date", "Last Update",
]

geo_time_profile = pd.DataFrame(
    {
        "Spalte": geo_time_cols,
        "Nicht Null (%)": [round(emdat_raw[c].notna().mean() * 100, 1) for c in geo_time_cols],
        "Einzigartige Werte": [int(emdat_raw[c].nunique(dropna=True)) for c in geo_time_cols],
        "Beispiel": [_sample_values(emdat_raw[c], n=2, max_len=70) for c in geo_time_cols],
    }
)
display(geo_time_profile)

print("\nISO zu Country, Beispiel Mapping:")
display(
    emdat_raw[["ISO", "Country"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["ISO", "Country"])
    .head(20)
)

print("\nMagnitude und Magnitude Scale, Top 12:")
for c in ["Magnitude", "Magnitude Scale"]:
    vc = emdat_raw[c].value_counts(dropna=False).head(12)
    print(f"\n{c}")
    display(vc.to_frame(name="Anzahl"))

,Spalte,Nicht Null (%),Einzigartige Werte,Beispiel
0,ISO,100.0,226,BRA | USA
1,Country,100.0,226,Brazil | United States of America
2,Region,100.0,5,Americas | Africa
3,Subregion,100.0,17,Latin America and the Caribbean | Northern Ame...
4,Latitude,10.1,1844,-22.479 | 23.5
5,Longitude,10.1,1838,-44.095 | 96.3
6,River Basin,7.3,1435,Hirmand | Helmand
7,Start Year,100.0,34,2018 | 2002
8,Start Month,99.4,12,2.0 | 6.0
9,Start Day,89.6,31,14.0 | 8.0



ISO zu Country, Beispiel Mapping:


,ISO,Country
16,AFG,Afghanistan
174,AGO,Angola
4269,AIA,Anguilla
264,ALB,Albania
1937,ANT,Netherlands Antilles
1182,ARE,United Arab Emirates
481,ARG,Argentina
774,ARM,Armenia
13975,ASM,American Samoa
1932,ATG,Antigua and Barbuda



Magnitude und Magnitude Scale, Top 12:

Magnitude


,Anzahl
Magnitude,
NaN,16416
100.0,86
130.0,65
120.0,63
150.0,59
5.6,56
160.0,53
200.0,49
6.4,42



Magnitude Scale


,Anzahl
Magnitude Scale,
NaN,7978
Km2,5869
Kph,3431
m3,1251
Moment Magnitude,887
°C,645
Vaccinated,596


Der rohe EMDAT Datensatz enthält 47 Spalten. Die folgende Tabelle begründet für jede Spalte die Entscheidung.

| # | Spalte | Kategorie | Entscheidung | Begründung laut Output |
|---|---|---|---|---|
| 0 | DisNo. | Identifier | **Behalten** | Das Spaltenprofil zeigt ein typisches ID Format pro Ereignis. |
| 1 | Historic | Klassifikation | Drop | Der Klassifikations Output zeigt nur Yes und No, also keine zusätzliche Trennschärfe für die Analysefrage. |
| 2 | Classification Key | Klassifikation | Drop | Der Output zeigt interne Codes wie nat-hyd-flo-riv, fachlich reicht die Kombination aus Disaster Type und Disaster Subtype. |
| 3 | Disaster Group | Klassifikation | Drop | Der Output zeigt nur zwei Gruppen, Natural und Technological, das ist gröber als Type und Subtype. |
| 4 | Disaster Subgroup | Klassifikation | **Behalten** | Der Output zeigt differenzierte Gruppen wie Hydrological, Transport und Meteorological, das bringt relevante Zusatzstruktur. |
| 5 | Disaster Type | Klassifikation | **Behalten** | Der Output zeigt zentrale Ereignisklassen wie Flood, Storm und Water, das ist die primäre Filterdimension. |
| 6 | Disaster Subtype | Klassifikation | **Behalten** | Der Output zeigt fachliche Feinauflösung wie Riverine flood und Flash flood, das ergänzt Type sinnvoll. |
| 7 | External IDs | Referenz | Drop | Reine Referenzschlüssel, ohne direkten Mehrwert für die Analyse. |
| 8 | Event Name | Beschreibung | Drop | Der Output zeigt nur etwa 32.4 % Füllgrad und unstrukturierten Freitext, das ist für robuste Aggregationen ungeeignet. |
| 9 | ISO | Geografie | **Behalten** | Der Geo und Zeit Output zeigt 100 % Füllgrad und 226 eindeutige Codes, das ist der stabilste Länderfilter. |
| 10 | Country | Geografie | **Behalten** | Der Geo und Zeit Output zusammen mit ISO zu Country Mapping liefert gute Lesbarkeit für Reporting und Plausibilisierung. |
| 11 | Subregion | Geografie | Drop | Der Geo und Zeit Output zeigt nur 17 Klassen, für Filterung ist ISO präziser und konsistenter. |
| 12 | Region | Geografie | **Behalten** | Der Geo und Zeit Output zeigt fünf Weltregionen und bietet kompakten Kontext für Auswertungen. |
| 13 | Location | Geografie | Drop | Der Text Output zeigt hohen Freitextanteil mit mittlerer Textlänge um 60 Zeichen, heterogen und schwer zu standardisieren. |
| 14 | Origin | Ereignis | **Behalten** | Der Klassifikations Output zeigt unterscheidbare Auslöser wie Heavy rain und Monsoonal rain, das ist potenziell analytisch relevant. |
| 15 | Associated Types | Klassifikation | Drop | Der Output zeigt starke Fragmentierung, viele Missing Values und kombinierte Mehrfachtags, dadurch geringe Modellierbarkeit. |
| 16 | OFDA/BHA Response | Verwaltung | Drop | Das Spaltenprofil zeigt ein administratives Antwortfeld, kein direkter physikalischer Treiber und kein Impact Kernmerkmal. |
| 17 | Appeal | Verwaltung | Drop | Das Spaltenprofil zeigt Verwaltungs und Prozessinformation, ausserhalb der fachlichen Kernfrage. |
| 18 | Declaration | Verwaltung | Drop | Das Spaltenprofil zeigt administrative Deklaration statt Ereignismerkmal, für Join und Impact nicht nötig. |
| 19 | AID Contribution | Wirtschaft | Drop | Das Spaltenprofil zeigt finanzielle Hilfsmetrik in USD, der Fokus liegt auf Ereignismerkmalen und Betroffenheit. |
| 20 | Magnitude | Physik | Drop | Der Magnitude Output zeigt viele Missing Values, zusammen mit heterogenen Skalen nur eingeschränkt vergleichbar. |
| 21 | Magnitude Scale | Physik | Drop | Wird mit Magnitude gedropped. |
| 22 | Latitude | Geografie | **Behalten** | Der Geo und Zeit Output zeigt trotz etwa 10.1 % Füllgrad viele eindeutige Koordinaten und damit hohe räumliche Auflösung. |
| 23 | Longitude | Geografie | **Behalten** | Gleicher Nachweis wie bei Latitude, punktgenaue Georeferenz für spätere räumliche Subsets. |
| 24 | River Basin | Geografie | Drop | Der Geo und Zeit Output zeigt nur etwa 7.3 % Füllgrad und ist irrelevant. |
| 25 | Start Year | Zeit | **Behalten** | Der Geo und Zeit Output zeigt 100 % Füllgrad, zentraler Zeitanker für Zeitreihen und Join. |
| 26 | Start Month | Zeit | **Behalten** | Der Geo und Zeit Output zeigt etwa 99.4 % Füllgrad, wichtig für monatliche Aggregationen und Join Optionen. (Falls wir das schlussendlich so aggregiert joinen) |
| 27 | Start Day | Zeit | **Behalten** | Der Geo und Zeit Output zeigt etwa 89.6 % Füllgrad, erhält Detailtiefe auf Tagesebene. |
| 28 | End Year | Zeit | **Behalten** | Der Geo und Zeit Output zeigt 100 % Füllgrad, nötig für Dauer und Intervallinterpretation. |
| 29 | End Month | Zeit | **Behalten** | Der Geo und Zeit Output zeigt etwa 98.7 % Füllgrad, konsistent mit Start Month. |
| 30 | End Day | Zeit | **Behalten** | Der Geo und Zeit Output zeigt etwa 90.1 % Füllgrad, ergänzt die zeitliche Vollständigkeit. |
| 31 | Total Deaths | Impact | **Behalten** | Der Impact Output zeigt hohe Verfügbarkeit von etwa 80.5 % und eine direkte Kernmetrik für den Schweregrad. |
| 32 | No. Injured | Impact | Drop | Der Impact Output zeigt, dass die Summe der Teilkomponenten dem Wert von Total Affected in 100 % der verfügbaren Fälle entspricht. |
| 33 | No. Affected | Impact | Drop | Laut Impact Output ist die Spalte bereits vollständig in Total Affected enthalten. |
| 34 | No. Homeless | Impact | Drop | Laut Impact Output ist die Spalte Teilkomponente von Total Affected.  |
| 35 | Total Affected | Impact | **Behalten** | Der Impact Output zeigt eine aggregierte Kennzahl, exakt gleich der Summe aus Injured, Affected und Homeless in den verfügbaren Fällen. |
| 36 | Reconstruction Costs | Wirtschaft | Drop | Das Spaltenprofil zeigt ökonomische Schadensmetrik, ausserhalb des aktuellen Analysefokus. |
| 37 | Reconstruction Costs, Adjusted | Wirtschaft | Drop | Das Spaltenprofil zeigt die inflationsbereinigte Variante derselben Kostendimension, damit redundant bei Drop der Basisspalte. |
| 38 | Insured Damage | Wirtschaft | Drop | Das Spaltenprofil zeigt Versicherungsschaden statt Expositions und Ereignismerkmal. |
| 39 | Insured Damage, Adjusted | Wirtschaft | Drop | Das Spaltenprofil zeigt die inflationsbereinigte Duplikatdimension zu Insured Damage. |
| 40 | Total Damage | Wirtschaft | Drop | Das Spaltenprofil zeigt monetären Gesamtschaden, nicht Kernziel der aktuellen Fragestellung. |
| 41 | Total Damage, Adjusted | Wirtschaft | Drop | Das Spaltenprofil zeigt die inflationsbereinigte Variante von Total Damage, für den Scope redundant. |
| 42 | CPI | Wirtschaft | Drop | Das Spaltenprofil zeigt Preisindex Korrekturfaktor, nur relevant falls ökonomische Schadensspalten modelliert werden. |
| 43 | Admin Units | Geografie | Drop | Der Output zeigt komplexe JSON Strukturen und lange Inhalte, für den aktuellen Scope zu aufwendig ohne Zusatznutzen. |
| 44 | GADM Admin Units | Geografie | Drop | Der Output zeigt sehr grosse JSON Grenzobjekte mit deutlich längeren Inhalten als andere Felder, nicht nötig für die Basisanalyse. |
| 45 | Entry Date | Metadaten | Drop | Der Geo und Zeit Output zeigt vollständig gefüllte Erfassungsmetadaten, das ist Datenbankprozess statt Ereignisinhalt. |
| 46 | Last Update | Metadaten | Drop | Der Geo und Zeit Output zeigt Revisionszeitpunkt als Metadatum, keine fachliche Ereignisvariable. |

**Resultat:** 18 Spalten werden behalten.

In [29]:
#| export
def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Nur angegebene Spalten behalten. Raises ValueError bei fehlenden Spalten."""
    missing = [c for c in columns if c not in df.columns]
    if missing:
        logger.error("Spaltenauswahl fehlgeschlagen; fehlende Spalten: %s", missing)
        raise ValueError(f"Spalten nicht vorhanden im DataFrame: {missing}")
    result = df[columns].copy()
    logger.info(
        "select_columns | cols: %d → %d",
        len(df.columns),
        len(columns),
    )
    return result

In [30]:
EMDAT_KEEP_COLS = [
    "DisNo.",
    "ISO",
    "Country",
    "Region",
    "Disaster Subgroup",
    "Disaster Type",
    "Disaster Subtype",
    "Origin",
    "Start Year",
    "Start Month",
    "Start Day",
    "End Year",
    "End Month",
    "End Day",
    "Latitude",
    "Longitude",
    "Total Affected",
    "Total Deaths",
]

emdat = select_columns(emdat_raw, EMDAT_KEEP_COLS)

2026-05-12 23:36:08 | myproj.transform     | INFO     | select_columns | cols: 47 → 18


In [31]:
print(f"EMDAT nach Spaltenauswahl: {emdat.shape[0]:,} Zeilen, {emdat.shape[1]} Spalten")
display(emdat.head(3))

EMDAT nach Spaltenauswahl: 20,657 Zeilen, 18 Spalten


,DisNo.,ISO,Country,Region,Disaster Subgroup,Disaster Type,Disaster Subtype,Origin,Start Year,Start Month,Start Day,End Year,End Month,End Day,Latitude,Longitude,Total Affected,Total Deaths
0,2018-0040-BRA,BRA,Brazil,Americas,Hydrological,Flood,Flood (General),Heavy rains,2018,2.0,14.0,2018,2.0,16.0,-22.479,-44.095,250.0,4.0
1,2002-0351-USA,USA,United States of America,Americas,Climatological,Wildfire,Forest fire,NaN,2002,6.0,8.0,2002,6.0,8.0,NaN,NaN,1572.0,NaN
2,1990-9604-BWA,BWA,Botswana,Africa,Climatological,Drought,Drought,NaN,1992,NaN,NaN,1992,NaN,NaN,NaN,NaN,100000.0,NaN


### Sea Level

Der Sea Level Datensatz enthält eine Zeitkoordinate und zwei Variablen.

In [32]:
data_cols = [c for c in sea_level_raw.columns if c != "time"]

sea_level_structure = pd.DataFrame(
    {
        "Variable": ["time"] + data_cols,
        "dtype": [str(sea_level_raw[c].dtype) for c in ["time"] + data_cols],
        "Nicht Null (%)": [
            round(sea_level_raw[c].notna().mean() * 100, 1)
            for c in ["time"] + data_cols
        ],
    }
)
display(sea_level_structure)

sea_level_stats = sea_level_raw[["time"] + data_cols].describe(include="all").T
display(sea_level_stats)

,Variable,dtype,Nicht Null (%)
0,time,datetime64[ns],100.0
1,MSL_filtered_GIA_corrected_adjusted,float32,100.0
2,trend_MSL_filtered_GIA_corrected_adjusted,float32,100.0


,count,mean,min,25%,50%,75%,max,std
time,9405,2012-01-05 00:00:00,1999-02-20 00:00:00,2005-07-29 00:00:00,2012-01-05 00:00:00,2018-06-13 00:00:00,2024-11-19 00:00:00,NaN
MSL_filtered_GIA_corrected_adjusted,9405.0,5.963125,-0.416788,3.405159,5.681014,8.220654,14.527944,3.293798
trend_MSL_filtered_GIA_corrected_adjusted,9405.0,5.963297,2.171483,3.535554,5.537774,8.178143,11.45666,2.707713


| Variable | Entscheidung | Begründung |
|---|---|---|
| time | **Behalten** | Zeitschlüssel für den Join |
| MSL_filtered_GIA_corrected_adjusted | **Behalten** | Meeresspiegelanomalie (GIA korrigiert, geglättet) in cm, Kernvariable |
| trend_MSL_filtered_GIA_corrected_adjusted | **Behalten** | Langzeittrend der Anomalie, potenziell nützlich für Trendanalyse |

Alle drei Felder werden behalten.

In [33]:
sea_level = sea_level_raw[["time", "MSL_filtered_GIA_corrected_adjusted", "trend_MSL_filtered_GIA_corrected_adjusted"]]
print(f"Sea Level nach Spaltenauswahl: {sea_level.shape[0]:,} Zeilen, {sea_level.shape[1]} Spalten")
display(sea_level.head(3))

Sea Level nach Spaltenauswahl: 9,405 Zeilen, 3 Spalten


,time,MSL_filtered_GIA_corrected_adjusted,trend_MSL_filtered_GIA_corrected_adjusted
0,1999-02-20,3.687602,2.171483
1,1999-02-21,3.684773,2.171927
2,1999-02-22,3.681582,2.172372


### Zusammenfassung Spalten entfernen

| Datensatz | Spalten vorher | Spalten nachher | Entfernt |
|---|---|---|---|
| EMDAT | 47 | 18 | 29 |
| Sea Level | 3 | 3 | 0 |

## Zeitliche Ausrichtung auf Sea-Level-Abdeckung

Der Sea-Level-Datensatz beginnt am **1999-02-20**. EMDAT-Einträge, die eindeutig vor diesem Datum einzuordnen sind, werden entfernt. Einträge, die sich nicht klar einordnen lassen (kein Monat, kein Tag), werden **behalten**.

### Grenzfälle
Es gibt aber gewisse Grenzfälle. Unten eine Tabelle mit den begründeten Entscheidungen zu diesen Grenzfällen.

## Zeitliche Ausrichtung auf Sea-Level-Abdeckung

Der Sea-Level-Datensatz hat eine definierte Abdeckung: **1999-02-20 bis 2024-11-19**. EMDAT-Einträge ausserhalb dieser Spanne können keinem Sea-Level-Wert zugeordnet werden und werden entfernt.

In [34]:
sl_min = pd.to_datetime(sea_level_raw["time"].min())
sl_max = pd.to_datetime(sea_level_raw["time"].max())
print(f"Sea-Level-Abdeckung: {sl_min.date()} – {sl_max.date()}")

Sea-Level-Abdeckung: 1999-02-20 – 2024-11-19


### Grenzfälle untere Grenze (1999-02-20)
Einträge aus 1999, bei denen sich nicht eindeutig bestimmen lässt, ob sie vor oder nach dem 20.02.1999 liegen.

In [35]:
in_1999 = emdat_raw["Start Year"] == 1999
cols = ["DisNo.", "Start Year", "Start Month", "Start Day", "Disaster Type", "Country"]

print("=== Grenzfall 1: Start Year = 1999, Start Month fehlt → behalten ===")
display(emdat_raw[in_1999 & emdat_raw["Start Month"].isna()][cols])

print("\n=== Grenzfall 2: Start Year = 1999, Monat = Feb, Tag fehlt → behalten ===")
display(emdat_raw[in_1999 & (emdat_raw["Start Month"] == 2) & emdat_raw["Start Day"].isna()][cols])

print("\n=== Grenzfall 3: Start Year = 1999, Monat = Feb, Tag < 20 → entfernen ===")
display(emdat_raw[in_1999 & (emdat_raw["Start Month"] == 2) & (emdat_raw["Start Day"] < 20)][cols])

=== Grenzfall 1: Start Year = 1999, Start Month fehlt → behalten ===


,DisNo.,Start Year,Start Month,Start Day,Disaster Type,Country
4395,1999-0651-IDN,1999,NaN,NaN,Epidemic,Indonesia
4418,1999-0674-KAZ,1999,NaN,NaN,Epidemic,Kazakhstan
4462,1999-0719-ZMB,1999,NaN,NaN,Epidemic,Zambia



=== Grenzfall 2: Start Year = 1999, Monat = Feb, Tag fehlt → behalten ===


,DisNo.,Start Year,Start Month,Start Day,Disaster Type,Country
3842,1999-0055-MEX,1999,2.0,NaN,Volcanic activity,Mexico
3843,1999-0056-HUN,1999,2.0,NaN,Storm,Hungary
3853,1999-0066-IRN,1999,2.0,NaN,Flood,Iran (Islamic Republic of)
3865,1999-0081-THA,1999,2.0,NaN,Flood,Thailand
3911,1999-0133-NIC,1999,2.0,NaN,Wildfire,Nicaragua
4222,1999-0476-ETH,1999,2.0,NaN,Epidemic,Ethiopia
4320,1999-0575-AFG,1999,2.0,NaN,Flood,Afghanistan
4484,1999-0741-PRY,1999,2.0,NaN,Epidemic,Paraguay



=== Grenzfall 3: Start Year = 1999, Monat = Feb, Tag < 20 → entfernen ===


,DisNo.,Start Year,Start Month,Start Day,Disaster Type,Country
3830,1999-0045-CHN,1999,2.0,14.0,Explosion (Industrial),China
3832,1999-0047-RUS,1999,2.0,2.0,Earthquake,Russian Federation
3834,1999-0049-AGO,1999,2.0,4.0,Air,Angola
3837,1999-0051-AUS,1999,2.0,2.0,Storm,Australia
3838,1999-0052-FRA,1999,2.0,9.0,Mass movement (wet),France
3841,1999-0054-AFG,1999,2.0,11.0,Earthquake,Afghanistan
3844,1999-0057-PAK,1999,2.0,18.0,Rail,Pakistan
3846,1999-0059-ZAF,1999,2.0,13.0,Road,South Africa
3849,1999-0062-CHN,1999,2.0,12.0,Road,China
3856,1999-0069-NGA,1999,2.0,18.0,Road,Nigeria


**Entscheid:** Alle Einträge aus 1999, die sich nicht eindeutig ab dem 20.02.1999 einordnen lassen, werden entfernt.

| Gruppe | Entscheid | Begründung |
|---|---|---|
| Start Year < 1999 | **Entfernen** | Eindeutig vor Sea-Level-Start |
| Start Year = 1999, Monat = Januar | **Entfernen** | Eindeutig vor 1999-02-20 |
| Start Year = 1999, Monat fehlt | **Entfernen** | Ausschliesslich Epidemien — kein Flood-Typ, für Analyse irrelevant |
| Start Year = 1999, Monat = Feb, Tag fehlt | **Entfernen** | Flood-Einträge ausserhalb Mittelmeerraum (IRN, THA, AFG) — für Analyse irrelevant |
| Start Year = 1999, Monat = Feb, Tag 1–19 | **Entfernen** | Eindeutig vor dem 20.02.1999 |

### Grenzfälle obere Grenze (2024-11-19)
Einträge aus 2024, bei denen sich nicht eindeutig bestimmen lässt, ob sie vor oder nach dem 19.11.2024 liegen.

In [36]:
in_end_year = emdat_raw["Start Year"] == sl_max.year
cols = ["DisNo.", "Start Year", "Start Month", "Start Day", "Disaster Type", "Country"]

print(f"=== Grenzfall 1: Start Year = {sl_max.year}, Monat fehlt ===")
display(emdat_raw[in_end_year & emdat_raw["Start Month"].isna()][cols])

print(f"\n=== Grenzfall 2: Start Year = {sl_max.year}, Monat = {sl_max.month}, Tag fehlt ===")
display(emdat_raw[
    in_end_year &
    (emdat_raw["Start Month"] == sl_max.month) &
    emdat_raw["Start Day"].isna()
][cols])

print(f"\n=== Grenzfall 3: Start Year = {sl_max.year}, Monat = {sl_max.month}, Tag > {sl_max.day} → entfernen ===")
display(emdat_raw[
    in_end_year &
    (emdat_raw["Start Month"] == sl_max.month) & 
    emdat_raw["Start Day"].notna() &
    (emdat_raw["Start Day"] > sl_max.day)
][cols])


=== Grenzfall 1: Start Year = 2024, Monat fehlt ===


,DisNo.,Start Year,Start Month,Start Day,Disaster Type,Country
13341,2024-0971-IND,2024,NaN,NaN,Miscellaneous accident (General),India



=== Grenzfall 2: Start Year = 2024, Monat = 11, Tag fehlt ===


,DisNo.,Start Year,Start Month,Start Day,Disaster Type,Country



=== Grenzfall 3: Start Year = 2024, Monat = 11, Tag > 19 → entfernen ===


,DisNo.,Start Year,Start Month,Start Day,Disaster Type,Country
5261,2024-0867-NLD,2024,11.0,26.0,Storm,Netherlands (Kingdom of the)
5309,2024-0955-BWA,2024,11.0,25.0,Storm,Botswana
5439,2024-0901-HTI,2024,11.0,30.0,Flood,Haiti
5444,2024-0961-GAB,2024,11.0,25.0,Flood,Gabon
13287,2024-0865-IDN,2024,11.0,23.0,Flood,Indonesia
13288,2024-0866-COD,2024,11.0,22.0,Mass movement (wet),Democratic Republic of the Congo
13289,2024-0867-FRA,2024,11.0,25.0,Storm,France
13290,2024-0867-GBR,2024,11.0,23.0,Storm,United Kingdom of Great Britain and Northern I...
13291,2024-0867-IRL,2024,11.0,23.0,Storm,Ireland
13292,2024-0870-MDG,2024,11.0,25.0,Water,Madagascar


**Entscheid:** Alle Einträge aus 2024, die sich nicht eindeutig bis zum 19.11.2024 einordnen lassen, werden entfernt.

| Gruppe | Entscheid | Begründung |
|---|---|---|
| Start Year > 2024 | **Entfernen** | Eindeutig nach Sea-Level-Ende |
| Start Year = 2024, Monat > 11 | **Entfernen** | Eindeutig nach 2024-11-19 |
| Start Year = 2024, Monat fehlt | **Entfernen** | Miscellaneous accident (General) — kein Flood-Typ, für Analyse irrelevant |
| Start Year = 2024, Monat = 11, Tag fehlt | — | Keine Einträge vorhanden |
| Start Year = 2024, Monat = 11, Tag > 19 | **Entfernen** | Flood-Einträge ausserhalb Mittelmeerraum (HTI, GAB, IDN, BOL, TZA, THA, MWI) — für Analyse irrelevant |


In [37]:
#| export
def filter_to_sea_level_coverage(
    df: pd.DataFrame,
    sea_level: pd.DataFrame,
    time_col: str = "time",
    year_col: str = "Start Year",
    month_col: str = "Start Month",
    day_col: str = "Start Day",
) -> pd.DataFrame:
    """
    Behält nur EMDAT-Einträge innerhalb der Sea-Level-Abdeckung.
    Einträge mit fehlenden Datumswerten werden verworfen, da sie nicht
    eindeutig der Abdeckungsperiode zugeordnet werden können.
    """
    s = pd.to_datetime(sea_level[time_col].min())
    e = pd.to_datetime(sea_level[time_col].max())

    date_lower = pd.to_datetime(pd.DataFrame({
        "year": df[year_col], "month": df[month_col].fillna(1), "day": df[day_col].fillna(1),
    }), errors="coerce")
    date_upper = pd.to_datetime(pd.DataFrame({
        "year": df[year_col], "month": df[month_col].fillna(12), "day": df[day_col].fillna(28),
    }), errors="coerce")

    mask = (date_lower >= s) & (date_upper <= e)
    result = df[mask].copy()
    logger.info(
        "filter_to_sea_level_coverage | rows: %d → %d | removed before: %d | removed after: %d | coverage: %s – %s",
        len(df), len(result),
        int((date_lower < s).sum()),
        int((date_upper > e).sum()),
        s.date(), e.date(),
    )
    return result

In [38]:
total = len(emdat_raw)
emdat = filter_to_sea_level_coverage(emdat_raw, sea_level_raw)

summary = pd.DataFrame({
    "Gruppe": ["Entfernt", "Verbleibend"],
    "Anzahl": [total - len(emdat), len(emdat)],
})
display(summary)


2026-05-12 23:36:08 | myproj.transform     | INFO     | filter_to_sea_level_coverage | rows: 20657 → 16699 | removed before: 3899 | removed after: 59 | coverage: 1999-02-20 – 2024-11-19


,Gruppe,Anzahl
0,Entfernt,3958
1,Verbleibend,16699
